# Modeling: 배터리 수명 예측 모델 개발 및 평가

설계된 Feature Sets를 바탕으로 점진적 비교 실험(Ablation), 규제 파라미터 튜닝, 후보 모델 비교를 거쳐 최적 모델을 확정하고 외부 일반화 성능 평가 수행

## 0. 필수 라이브러리 및 경로 설정

In [1]:
import json
import warnings
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.compose import TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import ElasticNet, Ridge
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

warnings.filterwarnings('ignore')
plt.rcParams['font.family'] = 'AppleGothic'
plt.rcParams['axes.unicode_minus'] = False

# 경로 설정
CURRENT_DIR = Path.cwd().resolve()
ROOT = CURRENT_DIR.parent if CURRENT_DIR.name in ('notebooks', 'test') else CURRENT_DIR
DATA_DIR = ROOT / 'results' / 'features'
OUTPUT_DIR = ROOT / 'results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'데이터 경로: {DATA_DIR}')
print(f'산출물 저장 경로: {OUTPUT_DIR}')

데이터 경로: /Users/yujin.je/Workspace/ds-mini/results/features
산출물 저장 경로: /Users/yujin.je/Workspace/ds-mini/results


## 1. 데이터 로드 및 평가 지표 정의
- **데이터 로드**: 02_feature_engineering에서 분할 저장된 Raw 데이터셋 및 `feature_sets.json` 로드
- **주요 평가 지표**: *Nature Energy* 원논문 벤치마크 기준인 MAPE(목표 9.1%) 산출 함수 정의
- **Valid 세트에 Hold-out 검증 방식을 채택한 이유**:
  - 셀 단위 독립성: 배터리 데이터는 셀 단위로 독립적이며 각 셀이 서로 다른 충전 프로토콜로 실험됨
  - 누수 차단: 무작위 교차 검증 적용 시 동일 프로토콜 셀이 Train과 Valid에 나뉘어 들어가 데이터 누수 위험 잔존
  - 일반화 평가 적합성: 충전 프로토콜 기준 분할을 통해 프로토콜 단위의 완전한 분리를 보장하고 배치 간 일반화를 엄밀히 평가하기에 더 적합

In [2]:
# 데이터셋 로드
train_df = pd.read_csv(DATA_DIR / 'train_features_raw.csv')
val_df = pd.read_csv(DATA_DIR / 'valid_features_raw.csv')
test_b2 = pd.read_csv(DATA_DIR / 'test_b2_features_raw.csv')
test_b3 = pd.read_csv(DATA_DIR / 'test_b3_features_raw.csv')

with open(DATA_DIR / 'feature_sets.json', 'r', encoding='utf-8') as f:
    FEATURE_SETS = json.load(f)

# 타겟 정의
y_train = train_df['cycle_life'].values
y_val = val_df['cycle_life'].values
y_b2 = test_b2['cycle_life'].values
y_b3 = test_b3['cycle_life'].values

# MAPE 산출 함수
def calculate_mape(y_true, y_pred):
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

print(f'Train: {len(train_df)}개 | Valid: {len(val_df)}개 | Test B2: {len(test_b2)}개 | Test B3: {len(test_b3)}개')
print(f'Feature Sets: {list(FEATURE_SETS.keys())}')

Train: 35개 | Valid: 11개 | Test B2: 39개 | Test B3: 44개
Feature Sets: ['A', 'B', 'C', 'D', 'ALT_delta_q_min', 'E_mean_QD']


## 2. Feature Sets 점진적 비교 실험 (Ablation)
- **실험 목적**: 피처 조합에 따른 교차 검증 오차(Train CV MAPE 및 CV Std)와 검증 오차(Valid MAPE) 비교
- **교차 검증**: 데이터 누수 방지를 위해 Train 세트 내부 `charging_policy` 기준 `GroupKFold(n_splits=4)` 교차 검증 적용
- **파이프라인 누수 방지**: 결측 대치 및 표준화 스케일러는 각 Fold 내부의 학습 데이터에만 `fit` 수행
- **테스트 세트 누수 배제**: 외부 테스트셋(Batch 2, Batch 3)은 모델 및 피처 선정 단계에서 철저히 배제

In [3]:
def evaluate_pipeline_cv(model, X, y, groups, n_splits=4):
    gkf = GroupKFold(n_splits=n_splits)
    scores = []
    for tr_idx, val_idx in gkf.split(X, y, groups=groups):
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        pipe.fit(X.iloc[tr_idx], y[tr_idx])
        pred = pipe.predict(X.iloc[val_idx])
        scores.append(calculate_mape(y[val_idx], pred))
    return np.mean(scores), np.std(scores)

ablation_results = []
for set_name, cols in FEATURE_SETS.items():
    pipe = Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('model', Ridge(alpha=1.0, random_state=42))
    ])
    
    cv_mean, cv_std = evaluate_pipeline_cv(Ridge(alpha=1.0, random_state=42), train_df[cols], y_train, train_df['charging_policy'])
    pipe.fit(train_df[cols], y_train)
    val_pred = pipe.predict(val_df[cols])
    
    ablation_results.append({
        'Feature Set': set_name,
        '피처 수': len(cols),
        'Train CV MAPE (%)': round(cv_mean, 2),
        'CV Std': round(cv_std, 2),
        'Valid MAPE (%)': round(calculate_mape(y_val, val_pred), 2),
    })

ablation_df = pd.DataFrame(ablation_results)
display(ablation_df)

,Feature Set,피처 수,Train CV MAPE (%),CV Std,Valid MAPE (%)
0,A,1,7.08,0.98,12.72
1,B,4,8.73,1.38,12.40
2,C,7,8.27,0.74,14.90
3,D,8,9.66,2.36,50.36
4,ALT_delta_q_min,1,8.89,1.98,9.41
5,E_mean_QD,9,8.99,3.44,42.48


### Feature Set 비교 분석 및 후보 선정
- **Set A (`log_delta_q_var`) 우선 선정**:
  - 가장 낮은 Train CV MAPE 7.08%를 기록했으며, CV Std도 0.98로 낮아 평균 성능과 안정성의 균형이 우수함
- **ALT_delta_q_min (`delta_q_min`) 대안 후보 유지**:
  - Hold-out Valid MAPE 9.41%로 내부 검증 세트에서 가장 낮은 오차를 기록하여 후속 튜닝 단계에서 비교 검토 대상으로 유지
- **다변수 Feature Sets(Set B, C, D, E) 미채택 사유**:
  - Set B: 충전 조건 피처를 추가했음에도 Valid MAPE 개선이 12.72%에서 12.40%로 제한적인 반면 Train CV MAPE는 7.08%에서 8.73%로 증가하여 추가 복잡도 대비 이점 제한적
  - Set C, D, E: 보조 피처 추가 후 CV 또는 Hold-out 성능이 악화되어 소표본 환경에서 일반화 성능이 불안정해짐을 확인

## 3. 후보 모델 및 하이퍼파라미터 튜닝
- **비교 후보군**: Ridge alpha 튜닝, ElasticNet 규제 조합 비교, Log-target 변환 효과 검증
- **평가 원칙**: 외부 테스트셋 누수 없이 Train CV(평균 및 Std)와 Hold-out Valid 성능만을 기준으로 최적 모델 확정

In [4]:
# 3-1. Ridge 규제 파라미터(alpha) 튜닝
tuning_results = []
for f_name in ['A', 'ALT_delta_q_min']:
    cols = FEATURE_SETS[f_name]
    for a in [0.01, 0.1, 1.0, 10.0, 100.0]:
        model = Ridge(alpha=a, random_state=42)
        cv_m, cv_s = evaluate_pipeline_cv(model, train_df[cols], y_train, train_df['charging_policy'])
        pipe = Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scaler', StandardScaler()),
            ('model', model)
        ])
        pipe.fit(train_df[cols], y_train)
        val_m = calculate_mape(y_val, pipe.predict(val_df[cols]))
        tuning_results.append({
            'Feature Set': f_name,
            'alpha': a,
            'Train CV MAPE (%)': round(cv_m, 2),
            'CV Std': round(cv_s, 2),
            'Valid MAPE (%)': round(val_m, 2),
        })

tuning_df = pd.DataFrame(tuning_results)
display(tuning_df)

,Feature Set,alpha,Train CV MAPE (%),CV Std,Valid MAPE (%)
0,A,0.01,7.18,1.15,13.14
1,A,0.10,7.17,1.13,13.10
2,A,1.00,7.08,0.98,12.72
3,A,10.00,8.83,1.73,9.77
4,A,100.00,16.21,3.08,11.84
5,ALT_delta_q_min,0.01,9.16,2.18,9.37
6,ALT_delta_q_min,0.10,9.13,2.16,9.37
7,ALT_delta_q_min,1.00,8.89,1.98,9.41
8,ALT_delta_q_min,10.00,8.90,1.65,10.37
9,ALT_delta_q_min,100.00,16.22,2.64,13.71


In [5]:
# 3-2. ElasticNet 규제 조합 비교
enet_results = []
for f_name in ['A', 'ALT_delta_q_min']:
    cols = FEATURE_SETS[f_name]
    for a in [0.01, 0.1, 1.0]:
        for l1 in [0.2, 0.5, 0.8]:
            model = ElasticNet(alpha=a, l1_ratio=l1, max_iter=10000, random_state=42)
            cv_m, cv_s = evaluate_pipeline_cv(model, train_df[cols], y_train, train_df['charging_policy'])
            pipe = Pipeline([
                ('imputer', SimpleImputer(strategy='median')),
                ('scaler', StandardScaler()),
                ('model', model)
            ])
            pipe.fit(train_df[cols], y_train)
            val_m = calculate_mape(y_val, pipe.predict(val_df[cols]))
            enet_results.append({
                'Feature Set': f_name,
                'alpha': a,
                'l1_ratio': l1,
                'Train CV MAPE (%)': round(cv_m, 2),
                'CV Std': round(cv_s, 2),
                'Valid MAPE (%)': round(val_m, 2),
            })

enet_df = pd.DataFrame(enet_results)
display(enet_df)

,Feature Set,alpha,l1_ratio,Train CV MAPE (%),CV Std,Valid MAPE (%)
0,A,0.01,0.2,7.16,1.12,13.02
1,A,0.01,0.5,7.17,1.13,13.07
2,A,0.01,0.8,7.18,1.14,13.11
3,A,0.10,0.2,7.10,0.90,12.02
4,A,0.10,0.5,7.09,0.95,12.42
5,A,0.10,0.8,7.12,1.06,12.84
6,A,1.00,0.2,10.89,2.40,7.73
7,A,1.00,0.5,9.48,2.03,8.46
8,A,1.00,0.8,7.74,1.29,10.55
9,ALT_delta_q_min,0.01,0.2,9.10,2.13,9.37


In [6]:
# 3-3. 타겟 변환 비교 (Raw Target vs Log Target)
target_exp_results = []
for f_name in ['A', 'ALT_delta_q_min']:
    cols = FEATURE_SETS[f_name]
    # Raw Target
    raw_model = Ridge(alpha=1.0, random_state=42)
    cv_raw_m, cv_raw_s = evaluate_pipeline_cv(raw_model, train_df[cols], y_train, train_df['charging_policy'])
    pipe_raw = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', raw_model)])
    pipe_raw.fit(train_df[cols], y_train)
    val_raw = calculate_mape(y_val, pipe_raw.predict(val_df[cols]))
    target_exp_results.append({
        'Feature Set': f_name,
        '타겟 변환': 'Raw Target',
        'Train CV MAPE (%)': round(cv_raw_m, 2),
        'CV Std': round(cv_raw_s, 2),
        'Valid MAPE (%)': round(val_raw, 2),
    })
    # Log Target
    log_model = TransformedTargetRegressor(regressor=Ridge(alpha=1.0, random_state=42), func=np.log, inverse_func=np.exp)
    cv_log_m, cv_log_s = evaluate_pipeline_cv(log_model, train_df[cols], y_train, train_df['charging_policy'])
    pipe_log = Pipeline([('imputer', SimpleImputer(strategy='median')), ('scaler', StandardScaler()), ('model', log_model)])
    pipe_log.fit(train_df[cols], y_train)
    val_log = calculate_mape(y_val, pipe_log.predict(val_df[cols]))
    target_exp_results.append({
        'Feature Set': f_name,
        '타겟 변환': 'Log Target',
        'Train CV MAPE (%)': round(cv_log_m, 2),
        'CV Std': round(cv_log_s, 2),
        'Valid MAPE (%)': round(val_log, 2),
    })

target_exp_df = pd.DataFrame(target_exp_results)
display(target_exp_df)

,Feature Set,타겟 변환,Train CV MAPE (%),CV Std,Valid MAPE (%)
0,A,Raw Target,7.08,0.98,12.72
1,A,Log Target,7.54,1.19,17.40
2,ALT_delta_q_min,Raw Target,8.89,1.98,9.41
3,ALT_delta_q_min,Log Target,7.80,1.58,9.47


### 모델 및 Feature Set 최종 확정 결론
- **Set A (`log_delta_q_var`) + Ridge(alpha=1.0) 우선 채택**:
  - Raw Target 기준 Train CV MAPE 7.08%, CV Std 0.98로 전체 후보 중 가장 낮은 평균 교차검증 오차 기록
  - Hold-out Valid MAPE는 12.72%로 ALT 계열보다 높았으나, 소표본 환경에서 특정 Hold-out 결과보다 GroupKFold 평균 성능과 Fold 간 안정성을 우선하여 Set A 선택
- **ALT_delta_q_min 대안 후보 유지**:
  - Log Target 적용 시 Train CV MAPE 7.80%, CV Std 1.58, Valid MAPE 9.47%로 Raw Target 대비 교차검증 성능이 크게 개선됨 확인
  - Valid 성능은 Set A보다 우수하지만 CV 평균과 변동성은 Set A보다 불리하므로 대안 후보로 유지
- **Ridge(alpha=1.0) 선택 사유**:
  - ElasticNet은 일부 조합에서 Hold-out 성능이 개선되었으나 CV 평균 또는 변동성이 악화되어 일관된 개선으로 판단하기 어려움
  - Set A에서 Log Target 적용 시 Valid MAPE가 12.72%에서 17.40%로 악화되어 Raw Target 최종 채택

## 4. 최종 모델 학습 및 성능 리포팅
- 확정된 최종 모델 파이프라인(Set A + StandardScaler + Ridge alpha=1.0) 학습
- `pjt-info.md` 표준 서식에 맞춘 Train, Valid, Test B2, Test B3 성능 및 Gap 산출 및 CSV 저장

In [7]:
selected_features = FEATURE_SETS['A']
TARGET_MAPE = 9.1

# 최종 최적 파이프라인 구축 및 학습
final_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler()),
    ('model', Ridge(alpha=1.0, random_state=42))
])
final_pipeline.fit(train_df[selected_features], y_train)

# 예측값 산출
train_cv_mape, train_cv_std = evaluate_pipeline_cv(Ridge(alpha=1.0, random_state=42), train_df[selected_features], y_train, train_df['charging_policy'])
valid_pred = final_pipeline.predict(val_df[selected_features])
b2_pred = final_pipeline.predict(test_b2[selected_features])
b3_pred = final_pipeline.predict(test_b3[selected_features])

valid_mape = calculate_mape(y_val, valid_pred)
test_b2_mape = calculate_mape(y_b2, b2_pred)
test_b3_mape = calculate_mape(y_b3, b3_pred)

# 공식 성능 리포팅 테이블 구성
report_rows = [
    {'구분': 'Train (Batch 1 CV)', 'MAPE (%)': f'{train_cv_mape:.2f}', '비고': ''},
    {'구분': 'Valid (Batch 1 Hold-out)', 'MAPE (%)': f'{valid_mape:.2f}', '비고': ''},
    {'구분': 'Test (Batch 2)', 'MAPE (%)': f'{test_b2_mape:.2f}', '비고': ''},
    {'구분': '  Gap (Train-Valid)', 'MAPE (%)': f'{valid_mape - train_cv_mape:+.2f}', '비고': '(+) : 과적합 의심'},
    {'구분': '  Gap (Valid-Test)', 'MAPE (%)': f'{test_b2_mape - valid_mape:+.2f}', '비고': '(+) : 배치간 일반화 저하 의심'},
    {'구분': '  Gap (Target-Test)', 'MAPE (%)': f'{test_b2_mape - TARGET_MAPE:+.2f}', '비고': 'Target : 원논문 9.1%'},
    {'구분': 'Test (Batch 3)', 'MAPE (%)': f'{test_b3_mape:.2f}', '비고': ''},
    {'구분': '  Gap (Batch2-Batch3)', 'MAPE (%)': f'{test_b3_mape - test_b2_mape:+.2f}', '비고': '(+) : Batch 3 성능 저하'},
    {'구분': '  Gap (Target-Test)', 'MAPE (%)': f'{test_b3_mape - TARGET_MAPE:+.2f}', '비고': 'Batch 3 기준, 원논문 9.1% 비교'},
]

report_df = pd.DataFrame(report_rows)
display(report_df)

report_df.to_csv(OUTPUT_DIR / 'model_performance.csv', index=False, encoding='utf-8-sig')
print(f'성능 리포트 저장 완료: {OUTPUT_DIR / "model_performance.csv"}')

,구분,MAPE (%),비고
0,Train (Batch 1 CV),7.08,
1,Valid (Batch 1 Hold-out),12.72,
2,Test (Batch 2),25.85,
3,Gap (Train-Valid),+5.64,(+) : 과적합 의심
4,Gap (Valid-Test),+13.12,(+) : 배치간 일반화 저하 의심
5,Gap (Target-Test),+16.75,Target : 원논문 9.1%
6,Test (Batch 3),12.16,
7,Gap (Batch2-Batch3),-13.69,(+) : Batch 3 성능 저하
8,Gap (Target-Test),+3.06,"Batch 3 기준, 원논문 9.1% 비교"


성능 리포트 저장 완료: /Users/yujin.je/Workspace/ds-mini/results/model_performance.csv
